In [19]:
from sklearn.metrics import accuracy_score
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
from elo import ELO

In [20]:
splits = joblib.load(r".\splits\8_windows_all_features.pkl")

In [21]:
random_proba, favourite_proba, h2h_proba, elo_proba = [], [], [], []
actuals = []

for i, split in enumerate(splits):
    X_train, y_train = split["X_train_raw"], split["y_train"]
    X_val, y_val = split["X_val_raw"], split["y_val"]
    X_test, y_test = split["X_test_raw"], split["y_test"]

    n_test = len(y_test)

    elo_tracker = ELO(k=32, initial=1500)

    for (_, row), y_true in zip(X_train.iterrows(), y_train):
        elo_tracker.update(row['team1_id'], row['team2_id'], float(y_true))

    for _, row in X_test.iterrows():
        prob = elo_tracker.predict(row['team1_id'], row['team2_id'])
        elo_proba.append(prob)

    random_proba.extend(np.random.uniform(0, 1, size=n_test))

    fav = (X_test['team1_match_win_rate'] > X_test['team2_match_win_rate']).astype(float)
    favourite_proba.extend(fav)

    h2h = (X_test['team1_match_h2h_win_rate'] > X_test['team2_match_h2h_win_rate']).astype(float)
    h2h_proba.extend(h2h)

    actuals.extend(y_test)

actuals = np.asarray(actuals)

def summarize(name, proba, eps=0.01):
    proba = np.clip(np.asarray(proba), eps, 1 - eps)  # avoid log(0) for hard 0/1 baselines
    pred = (proba > 0.5).astype(int)
    return {
        "model": name,
        "n": len(actuals),
        "accuracy": accuracy_score(actuals, pred),
        "log_loss": log_loss(actuals, proba, labels=[0, 1]),
        "brier": brier_score_loss(actuals, proba),
    }

benchmark_df = pd.DataFrame([
    summarize("Random", random_proba),
    summarize("Favourite (win rate)", favourite_proba),
    summarize("H2H", h2h_proba),
    summarize("Elo", elo_proba),
])
print(benchmark_df)

                  model    n  accuracy  log_loss     brier
0                Random  656  0.535061  0.904937  0.309971
1  Favourite (win rate)  656  0.574695  1.964377  0.416899
2                   H2H  656  0.510671  2.258577  0.479643
3                   Elo  656  0.524390  0.837591  0.295109
